In [1]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score



In [2]:
# Paths to external model predictions (these may not exist in the environment)
path_bart = "/kaggle/input/toxic-torii-v5-bart-ver1-result/submission.csv"
path_tfidf = "/kaggle/input/toxic-torii-v3-tfidf-ver1-result/submission.csv"
path_gru = "/kaggle/input/toxic-torii-v8-gru-ver1-result/submission_gru.csv"
path_lstm = "/kaggle/input/toxic-torii-v9-lstm-ver1-result/submission_lstm.csv"
path_feature = "/kaggle/input/toxic-torii-v7-featureengineering2-ver1-result/submission_andre_features_enhanced.csv"

print("Loading external model predictions...")


def load_submission(path, name):
    if os.path.exists(path):
        print(f" - {name}: loaded successfully ({path})")
        return pd.read_csv(path)
    else:
        print(f" - {name}: file not found ({path})")
        return None


df_bart = load_submission(path_bart, "BART")
df_tfidf = load_submission(path_tfidf, "TF-IDF")
df_gru = load_submission(path_gru, "GRU")
df_lstm = load_submission(path_lstm, "LSTM")
df_feature = load_submission(path_feature, "FeatureEngineering")

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]



Loading external model predictions...
 - BART: file not found (/kaggle/input/toxic-torii-v5-bart-ver1-result/submission.csv)
 - TF-IDF: file not found (/kaggle/input/toxic-torii-v3-tfidf-ver1-result/submission.csv)
 - GRU: file not found (/kaggle/input/toxic-torii-v8-gru-ver1-result/submission_gru.csv)
 - LSTM: file not found (/kaggle/input/toxic-torii-v9-lstm-ver1-result/submission_lstm.csv)
 - FeatureEngineering: file not found (/kaggle/input/toxic-torii-v7-featureengineering2-ver1-result/submission_andre_features_enhanced.csv)


In [3]:
# Load the sample submission as a template
sample_submission_path = (
    "../input/jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip"
)
submission_df = pd.read_csv(sample_submission_path)

# Flag indicating whether we have a full ensemble to blend
have_all_models = all(
    df is not None for df in [df_bart, df_tfidf, df_gru, df_lstm, df_feature]
)

if have_all_models:
    print("\nAll external predictions found – blending them.")
    blend_preds = (
        df_bart[label_cols] * 0.20
        + df_tfidf[label_cols] * 0.20
        + df_gru[label_cols] * 0.20
        + df_lstm[label_cols] * 0.20
        + df_feature[label_cols] * 0.20
    )
    submission_df[label_cols] = blend_preds
else:
    print(
        "\nMissing one or more external predictions – training a simple TF‑IDF + LogisticRegression baseline."
    )
    # Load training data
    train_path = "../input/jigsaw-toxic-comment-classification-challenge/train.csv"
    train_df = pd.read_csv(train_path)
    X = train_df["comment_text"].fillna("")
    y = train_df[label_cols]

    # Split just to obtain a quick validation AUC (optional, not required for submission)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.1, random_state=42, stratify=y["toxic"]
    )

    # TF‑IDF vectorizer
    tfidf = TfidfVectorizer(
        max_features=100_000, ngram_range=(1, 2), stop_words="english", dtype="float32"
    )
    X_train_vec = tfidf.fit_transform(X_train)
    X_val_vec = tfidf.transform(X_val)

    # One‑vs‑Rest Logistic Regression
    ovr = OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear", C=4.0, max_iter=200, class_weight="balanced", n_jobs=-1
        )
    )
    ovr.fit(X_train_vec, y_train)

    # Optional validation AUC print (helps gauge distance to target)
    val_pred = ovr.predict_proba(X_val_vec)
    aucs = [
        roc_auc_score(y_val[col], val_pred[:, i]) for i, col in enumerate(label_cols)
    ]
    print("Validation AUCs:", dict(zip(label_cols, aucs)))
    print("Mean validation AUC:", sum(aucs) / len(aucs))

    # Train on full data
    X_full_vec = tfidf.fit_transform(train_df["comment_text"].fillna(""))
    ovr.fit(X_full_vec, y)

    # Predict on test set
    test_path = "../input/jigsaw-toxic-comment-classification-challenge/test.csv"
    test_df = pd.read_csv(test_path)
    X_test_vec = tfidf.transform(test_df["comment_text"].fillna(""))
    test_pred = ovr.predict_proba(X_test_vec)

    # Fill submission dataframe
    for i, col in enumerate(label_cols):
        submission_df[col] = test_pred[:, i]

# Write the final submission file
output_path = "submission_final_ensemble.csv"
submission_df.to_csv(output_path, index=False)
print(f"\nSubmission file written to {output_path}")
print("First few rows of the submission:")
print(submission_df.head())


Missing one or more external predictions – training a simple TF‑IDF + LogisticRegression baseline.


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:2072: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any ef

Validation AUCs: {'toxic': 0.9737021552183464, 'severe_toxic': 0.9837122342595608, 'obscene': 0.9843617287171349, 'threat': 0.9789577571033441, 'insult': 0.9784836802348595, 'identity_hate': 0.9774136587040173}
Mean validation AUC: 0.9794385357062104


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:2072: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1211: UserWarning: 'n_jobs' > 1 does not have any ef


Submission file written to submission_final_ensemble.csv
First few rows of the submission:
                 id     toxic  severe_toxic   obscene    threat    insult  \
0  00001cee341fdb12  0.999989      0.806889  0.999995  0.251199  0.997699   
1  0000247867823ef7  0.003843      0.002091  0.001176  0.002733  0.006299   
2  00013b17ad220c46  0.050080      0.006463  0.020078  0.001524  0.033762   
3  00017563c3f7919a  0.006158      0.006509  0.002595  0.001238  0.002047   
4  00017695ad8997eb  0.033878      0.008680  0.013911  0.005587  0.035433   

   identity_hate  
0       0.816996  
1       0.005239  
2       0.014747  
3       0.000472  
4       0.006753  
